# v72 — sequence representation gate

**Status:** executed, negative result. **Anchor:** `agas/v66/output/submission_v66_04_ext0.5_tm30_lgb_lean.csv` (public 0.56424).

v71 `core71` scored 0.56111 on public even though its OOF KS was +0.004 above the
anchor. v72 therefore tested a genuinely different representation: the last 24
calendar months of account cash flow, repayment, e-commerce, bureau reports, and
previous applications as a sequence. Every event is cut at `decision_datetime`.

The notebook is reproducible with `RUN_TRAINING=False`. It reads the stored
tensor, predictions, and summaries; it does not retrain the GRU.

## 0. Setup

In [1]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, rankdata, spearmanr
from sklearn.metrics import roc_auc_score

RUN_TRAINING = False
ROOT = Path.cwd()
while not (ROOT / "agas/shared/paths.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

OUT = ROOT / "agas/v72/output"
print("RUN_TRAINING:", RUN_TRAINING)
print("ROOT:", ROOT)
print("output exists:", OUT.exists())

RUN_TRAINING: False
ROOT: /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026
output exists: True


## 1. Data representation

`sequences_v72.npz` has shape `(100000, 24, 56)` and five source masks. The
channels are grouped as account (10), repayment (13), e-commerce (16), bureau
(8), and previous applications (9). A month with no event for a source has mask
0, so the model can distinguish "zero" from "not observed".

In [2]:
seq = np.load(OUT / "sequences_v72.npz", allow_pickle=True)
X, M = seq["X"], seq["M"]
channels = seq["channels"].tolist()
print("X:", X.shape, "M:", M.shape, "dtype:", X.dtype)
print("apps:", len(seq["application_id"]), "train:", int(np.isfinite(seq["y"]).sum()),
      "test:", int((~np.isfinite(seq["y"])).sum()))
print("channels:", len(channels))
print(channels)
for i, name in enumerate(["account", "repayment", "ecommerce", "bureau", "previous"]):
    print(f"  {name:10s} mask mean={M[:, :, i].mean():.3f}")

X: (100000, 24, 56) M: (100000, 24, 5) dtype: float32
apps: 100000 train: 70000 test: 30000
channels: 56
['acct_in', 'acct_out', 'acct_inc', 'acct_net_inc', 'acct_bal', 'acct_min', 'acct_close', 'acct_low', 'acct_out_in', 'acct_inc_share', 'rep_due', 'rep_paid', 'rep_unpaid', 'rep_n', 'rep_loans', 'rep_late', 'rep_late30', 'rep_late90', 'rep_dpd', 'rep_dpd_now', 'rep_open', 'rep_payratio', 'rep_due_inc', 'ec_spend', 'ec_orders', 'ec_cancel', 'ec_max_order', 'ec_items', 'ec_lrel', 'ec_need', 'ec_entropy', 'ec_spend_inc', 'ec_share_education', 'ec_share_electronics', 'ec_share_fashion', 'ec_share_groceries', 'ec_share_health', 'ec_share_home', 'ec_share_personal_care', 'bu_score', 'bu_missing', 'bu_util', 'bu_inq', 'bu_delinq', 'bu_active', 'bu_out', 'bu_rev', 'prev_n', 'prev_appr', 'prev_rej', 'prev_can', 'prev_req', 'prev_amt', 'prev_rate', 'prev_inst', 'prev_lag']
  account    mask mean=0.452
  repayment  mask mean=0.380
  ecommerce  mask mean=0.520
  bureau     mask mean=0.228
  prev

## 2. Point-in-time audit

The sequence builder keeps only events at or before the decision. This cell
rechecks the raw sources directly. All four "future" counts must be zero.

In [3]:
dt = ["application_datetime", "decision_datetime"]
tr = pd.read_csv(ROOT / "data/train_applications.csv", parse_dates=dt)
te = pd.read_csv(ROOT / "data/test_applications.csv", parse_dates=dt)
cur = pd.concat([
    tr[["application_id", "customer_id", "decision_datetime"]],
    te[["application_id", "customer_id", "decision_datetime"]],
], ignore_index=True)
print("target prevalence:", round(tr.target.mean(), 6))

rh = pd.read_csv(ROOT / "data/repayment_history.csv", parse_dates=["due_date"])
rh = rh.merge(cur[["customer_id", "decision_datetime"]], on="customer_id", how="left")
print("repayment due after decision:", int((rh.due_date > rh.decision_datetime).sum()))

ab = pd.read_csv(ROOT / "data/account_behavior.csv", parse_dates=["month"])
ab = ab.merge(cur[["customer_id", "decision_datetime"]], on="customer_id", how="left")
print("account month after decision:", int((ab.month > ab.decision_datetime).sum()))

ec = pd.read_csv(ROOT / "data/ecommerce_transactions.csv",
                 usecols=["customer_id", "transaction_datetime"],
                 parse_dates=["transaction_datetime"])
ec = ec.merge(cur[["customer_id", "decision_datetime"]], on="customer_id", how="left")
print("ecommerce transaction after decision:", int((ec.transaction_datetime > ec.decision_datetime).sum()))

bu = pd.read_csv(ROOT / "data/bureau_observations.csv",
                 usecols=["customer_id", "available_datetime"],
                 parse_dates=["available_datetime"])
bu = bu.merge(cur[["customer_id", "decision_datetime"]], on="customer_id", how="left")
print("bureau report available after decision:", int((bu.available_datetime > bu.decision_datetime).sum()))

pa = pd.read_csv(ROOT / "data/previous_applications.csv", usecols=["application_id"])
print("current application ids found in previous_applications:",
      len(set(tr.application_id) & set(pa.application_id)))

target prevalence: 0.135814


repayment due after decision: 0


account month after decision: 0


ecommerce transaction after decision: 0


bureau report available after decision: 0
current application ids found in previous_applications: 0


## 3. Sequence legs

Both runs use the same 5-fold stratified split (fold seed 42, model seed 0).
`sequence + static` appends the 770 numeric v70 columns, so it is the fair
comparison against the existing static family. The pure sequence leg is the
decorrelation probe.

In [4]:
legs = {}
for name, fname in [("sequence", "preds_seq_v72_pure.npz"),
                    ("sequence+static", "preds_seq_v72_static.npz")]:
    z = np.load(OUT / fname, allow_pickle=True)
    y = z["y"].astype(int)
    p = z["oof_42"]
    legs[name] = (y, p)
    print(f"{name:16s} AUC={roc_auc_score(y, p):.6f} KS={ks_2samp(p[y == 1], p[y == 0]).statistic:.6f}")

sequence         AUC=0.783309 KS=0.407868
sequence+static  AUC=0.841527 KS=0.514382


## 4. Blend gate against the anchor

The anchor is rebuilt from the stored v66 legs using exactly the v66_04 chain.
A candidate must add KS, not only AUC. The table uses rank blends so the scale
of each leg is irrelevant.

In [5]:
z66 = np.load(ROOT / "agas/v66/output/preds_v66.npz", allow_pickle=True)
train_ids = z66["train_ids"]
y = pd.read_csv(ROOT / "data/train_applications.csv",
                usecols=["application_id", "target"]
                ).set_index("application_id").loc[train_ids].target.values.astype(int)
rank01 = lambda v: rankdata(v) / len(v)
ro = {k[4:]: rank01(z66[k]) for k in z66.files if k.startswith("oof_")}
trees = (ro["lgb_full"] + ro["lgb_lean"] + ro["cat_full"] + ro["xgb_full"]) / 4
core = (2 * trees + ro["lgbx_full"]) / 3
tm30 = 0.7 * core + 0.3 * ro["linear"]
anchor = rank01(tm30 + 0.5 * (tm30 - ro["lgb_lean"]))
print(f"anchor AUC={roc_auc_score(y, anchor):.6f} KS={ks_2samp(anchor[y == 1], anchor[y == 0]).statistic:.6f}")

for name, (_, p) in legs.items():
    print()
    print(name)
    for w in (0.0, 0.02, 0.05, 0.10, 0.15, 0.25):
        b = rank01((1 - w) * anchor + w * rank01(p))
        print(f"  w={w:0.2f}  AUC={roc_auc_score(y, b):.6f}  "
              f"KS={ks_2samp(b[y == 1], b[y == 0]).statistic:.6f}  "
              f"rho={spearmanr(b, anchor).statistic:.6f}")

anchor AUC=0.846840 KS=0.528713

sequence
  w=0.00  AUC=0.846840  KS=0.528713  rho=1.000000
  w=0.02  AUC=0.846937  KS=0.528632  rho=0.999916


  w=0.05  AUC=0.846968  KS=0.528844  rho=0.999469
  w=0.10  AUC=0.846722  KS=0.528193  rho=0.997834
  w=0.15  AUC=0.846122  KS=0.527034  rho=0.995026


  w=0.25  AUC=0.843867  KS=0.524433  rho=0.985623

sequence+static
  w=0.00  AUC=0.846840  KS=0.528713  rho=1.000000
  w=0.02  AUC=0.846992  KS=0.528106  rho=0.999980
  w=0.05  AUC=0.847197  KS=0.528573  rho=0.999877
  w=0.10  AUC=0.847488  KS=0.528205  rho=0.999504
  w=0.15  AUC=0.847708  KS=0.528063  rho=0.998879


  w=0.25  AUC=0.847959  KS=0.528276  rho=0.996858


## 5. Candidate files and decision

Only the anchor is recommended. The two sequence blends are diagnostics: they
are valid CSVs, but their OOF gain is inside the noise floor and their KS is not
higher than the anchor. The bootstrap column is `dKS` against the anchor on
2,000 stratified 30,000-row resamples.

In [6]:
summary = json.loads((OUT / "summary_v72_candidates.json").read_text())
print("decision:", summary["decision"])
print()
print(f"{'file':34s} {'w':>4s} {'AUC':>9s} {'KS':>9s} {'rho':>8s} valid rec")
for r in summary["candidates"]:
    d = pd.read_csv(OUT / r["file"])
    valid = (len(d) == 30000 and d.application_id.is_unique
             and np.isfinite(d.default_probability).all()
             and d.default_probability.between(0, 1).all())
    print(f"{r['file']:34s} {r['weight']:4.2f} {r['oof_auc']:9.6f} "
          f"{r['oof_ks']:9.6f} {r['spearman_anchor']:8.6f} {valid} {r['recommended']}")
print()
for name, b in summary["bootstrap_dks_vs_anchor"].items():
    print(f"{name}: dKS={b['mean_dks']:+.6f}, P(dKS>0)={b['p_gt0']:.3f}")

decision: no sequence candidate promoted; keep v66_04 anchor

file                                  w       AUC        KS      rho valid rec
submission_v72_01_anchor.csv       0.00  0.846840  0.528713 1.000000 True True
submission_v72_02_anchor_seq05.csv 0.05  0.846968  0.528844 0.999501 True False
submission_v72_03_anchor_seqst15.csv 0.15  0.847708  0.528063 0.995056 True False

02_anchor_seq05: dKS=+0.000519, P(dKS>0)=0.676
03_anchor_seqst15: dKS=+0.000287, P(dKS>0)=0.585


## 6. Conclusion

- The sequence is a real but weak signal: pure OOF AUC 0.7833, rank correlation
  only 0.76 to the anchor.
- Adding the static block raises it to 0.8415, still below the static MLP
  (0.8484), so the sequence branch is not a better static model.
- Blend weights that buy a small AUC increase do not buy KS. The 5% pure
  sequence blend has P(dKS>0) = 0.676, which is not evidence.
- **No v72 candidate is promoted.** Keep `v66_04` as the public anchor and
  move to the v73 freeze/verification plan in `agas/plan.md`.

Artifacts:

- `agas/v72/training/build_sequences_v72.py`
- `agas/v72/training/train_seq_v72.py`
- `agas/v72/training/make_submission_v72.py`
- `agas/v72/output/summary_v72_candidates.json`
- `agas/v72/output/submission_v72_01_anchor.csv`